# 02 — Fundamentos de PyTorch: un mini modelo de lenguaje carácter a carácter

**Pregunta que responde este notebook:** ¿qué pasa *de verdad* dentro de un bucle de entrenamiento? Sin librerías de entrenamiento, sin abstracciones: tensores, gradientes, `loss.backward()`, un optimizador y `zero_grad()`, escritos a mano y ejecutados sobre texto real del dataset de SkillGen.

Al final debes poder explicar, sin mirar:

1. Qué es un **tensor** y qué es un **gradiente**.
2. Por qué `loss.backward()` rellena el atributo `.grad` de cada parámetro.
3. Qué hace el optimizador con el *learning rate* (lr).
4. Por qué hay que poner los gradientes a cero en cada paso.
5. Qué aspecto tendría el **sobreajuste** (*overfitting*) en este juguete.

> Nota de entorno: aquí usamos el wheel de **CPU** de PyTorch (`torch` resuelto desde el índice `https://download.pytorch.org/whl/cpu` en `pyproject.toml`). En esta máquina no hay GPU y no la necesitamos para aprender los fundamentos. En Kaggle, la Fase 3 usará el torch con CUDA que la propia plataforma ya trae instalado: este pin de CPU solo afecta al desarrollo local.

## 1. Los datos: texto real de blueprints

Cargamos 3 registros del split de entrenamiento congelado (`data/splits/train_shard_00.jsonl`) y nos quedamos con los primeros 1.500 caracteres de cada `expected_blueprint`. Es poco texto a propósito: queremos un modelo que *memorice* patrones visibles en minutos de CPU, no un modelo bueno.

El modelo es **de nivel de carácter**: su vocabulario no son palabras ni tokens BPE, sino caracteres individuales. La tarea es siempre la misma: dados 32 caracteres, predecir el carácter 33.

In [1]:
import json
from pathlib import Path

import torch
import torch.nn.functional as F
from torch import nn

torch.manual_seed(7)  # reproducibilidad: mismos números en cada ejecución

# Localizar la raíz del repo subiendo desde el directorio del notebook.
root = Path.cwd()
while not (root / "data" / "splits").exists() and root != root.parent:
    root = root.parent

# --- Corpus: 3 blueprints reales, truncados a 1.500 caracteres cada uno ---
texts = []
with open(root / "data" / "splits" / "train_shard_00.jsonl", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        record = json.loads(line)
        blueprint = record["expected_blueprint"]
        if not isinstance(blueprint, str):
            blueprint = json.dumps(blueprint)
        texts.append(blueprint[:1500])

text = "\n".join(texts)
print(f"corpus: {len(text)} caracteres de {len(texts)} blueprints reales")
print(text[:200])

corpus: 4502 caracteres de 3 blueprints reales
{"blueprint_version": "1.0", "name": "platform-metadata-api-context-get", "description": "REQUIRED companion for Salesforce metadata generation — load this schema/API-context skill in the SAME turn as


## 2. Tensores: los datos ya son números

Un **tensor** es simplemente un array multidimensional de números con dos superpoderes: sabe en qué dispositivo vive (CPU/GPU) y, si se lo pedimos, **recuerda qué operaciones lo produjeron**. Esa memoria es lo que después permite calcular gradientes.

Aquí convertimos cada carácter en un entero (su índice en el vocabulario). El modelo nunca ve letras: ve enteros, los convierte en vectores (*embeddings*) y trabaja solo con números.

In [2]:
# Vocabulario: cada carácter distinto es un "token".
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}  # carácter -> entero
itos = {i: c for c, i in stoi.items()}     # entero -> carácter

# Todo el corpus como UN tensor 1-D de enteros.
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)

print(f"vocabulario: {vocab_size} caracteres distintos")
print(f"tensor de datos: forma {tuple(data.shape)}, dtype {data.dtype}")
print(f"primeros 20 enteros: {data[:20].tolist()}")
print(f"que decodifican a: {''.join(itos[int(i)] for i in data[:20])!r}")

vocabulario: 263 caracteres distintos
tensor de datos: forma (4502,), dtype torch.int64
primeros 20 enteros: [77, 2, 53, 63, 72, 56, 67, 69, 60, 65, 71, 50, 73, 56, 69, 70, 60, 66, 65, 2]
que decodifican a: '{"blueprint_version"'


## 3. El modelo: embedding + cabeza MLP

El modelo más pequeño que sigue siendo honesto:

1. **Embedding de token**: cada carácter (entero) se convierte en un vector de 64 números aprendidos.
2. **Embedding de posición**: otro vector aprendido dice "estás en la posición 7 de la ventana", porque un MLP no conoce el orden por sí solo.
3. **Cabeza MLP**: aplanamos la ventana (32 × 64 = 2.048 números) y la pasamos por `Linear → GELU → Linear` hasta obtener un *logit* por cada carácter posible del vocabulario.

Los logits no son probabilidades: son puntuaciones sin normalizar. La función de pérdida (entropía cruzada) se encarga de convertirlos en probabilidades y de castigar al modelo cuando el carácter correcto tenía una puntuación baja.

In [3]:
BLOCK = 32    # ventana de contexto: 32 caracteres para predecir el siguiente
N_EMBD = 64   # tamaño de cada embedding


class TinyCharLM(nn.Module):
    """Mini modelo de lenguaje: embedding + posición + cabeza MLP."""

    def __init__(self, vocab_size: int):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, N_EMBD)
        self.position_embedding = nn.Embedding(BLOCK, N_EMBD)
        self.head = nn.Sequential(
            nn.Linear(BLOCK * N_EMBD, 256),
            nn.GELU(),
            nn.Linear(256, vocab_size),
        )

    def forward(self, idx, targets=None):
        # idx: (batch, BLOCK) enteros. targets: (batch,) el carácter siguiente.
        positions = torch.arange(idx.shape[1], device=idx.device)
        x = self.token_embedding(idx) + self.position_embedding(positions)
        logits = self.head(x.flatten(1))  # (batch, vocab_size)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits, targets)
        return logits, loss


model = TinyCharLM(vocab_size)
n_params = sum(p.numel() for p in model.parameters())
print(f"parámetros del modelo: {n_params:,}")
for name, p in model.named_parameters():
    print(f"  {name:35s} {tuple(p.shape)}  requires_grad={p.requires_grad}")

parámetros del modelo: 611,015
  token_embedding.weight              (263, 64)  requires_grad=True
  position_embedding.weight           (32, 64)  requires_grad=True
  head.0.weight                       (256, 2048)  requires_grad=True
  head.0.bias                         (256,)  requires_grad=True
  head.2.weight                       (263, 256)  requires_grad=True
  head.2.bias                         (263,)  requires_grad=True


## 4. ¿Qué es un gradiente? (la idea en 30 segundos)

La pérdida (*loss*) es **un solo número** que resume "qué tan mal predice el modelo". Si movemos un parámetro un poquito, la pérdida sube o baja. El **gradiente** de un parámetro es exactamente eso: cuánto cambia la pérdida si ese parámetro aumenta una unidad, manteniendo todo lo demás fijo.

- Gradiente **grande y positivo** → ese parámetro está empujando la pérdida hacia arriba: hay que *bajarlo*.
- Gradiente **cercano a cero** → ese parámetro ahora mismo no importa mucho.

PyTorch calcula los gradientes de **todos** los parámetros a la vez con diferenciación automática: durante el `forward`, cada operación anota cómo derivarla; `loss.backward()` recorre ese grafo al revés (regla de la cadena) y deposita el resultado en el atributo `.grad` de cada parámetro. Por eso `backward()` no devuelve nada: su trabajo es *rellenar* los `.grad`.

El **optimizador** lee esos `.grad` y da un paso en la dirección que baja la pérdida: `parámetro ← parámetro − lr × gradiente` (AdamW, el que usamos aquí, además adapta el paso por parámetro usando medias móviles del gradiente). El **learning rate** es el tamaño de ese paso: demasiado pequeño y el aprendizaje es lentísimo; demasiado grande y el modelo salta por encima del mínimo y la pérdida explota.

¿Y `zero_grad()`? Porque PyTorch **acumula** gradientes: cada `backward()` *suma* sobre el `.grad` anterior. Si no limpiamos, el paso de hoy mezclará gradientes de ayer. (La acumulación es útil a propósito en otro contexto —*gradient accumulation*— pero aquí sería un bug silencioso.)

In [4]:
def get_batch(batch_size: int = 64):
    """Ventanas aleatorias del corpus: X = 32 caracteres, y = el carácter 33."""
    starts = torch.randint(len(data) - BLOCK - 1, (batch_size,))
    x = torch.stack([data[i : i + BLOCK] for i in starts])
    y = torch.stack([data[i + BLOCK] for i in starts])
    return x, y


# Demostración mínima de .grad: un forward, un backward, y miramos dentro.
xb, yb = get_batch(8)
_, loss = model(xb, yb)
print(f"loss antes de entrenar (batch de 8): {loss.item():.4f}")
print(f".grad antes de backward(): {model.head[0].weight.grad}")

loss.backward()
print(f".grad después de backward(): tensor {tuple(model.head[0].weight.grad.shape)}, "
      f"norma {model.head[0].weight.grad.norm().item():.4f}")
model.zero_grad()
# Ojo: zero_grad() en PyTorch moderno pone .grad a None (no a ceros):
# es más barato y significa lo mismo para el optimizador, "sin gradiente".
print(f".grad después de zero_grad(): {model.head[0].weight.grad}")

loss antes de entrenar (batch de 8): 5.7089
.grad antes de backward(): None
.grad después de backward(): tensor (256, 2048), norma 8.2673
.grad después de zero_grad(): None


## 5. EL BUCLE DE ENTRENAMIENTO MANUAL

Este es el centro del notebook. Todo entrenamiento en PyTorch —incluido el QLoRA de la Fase 3— es este mismo esqueleto con más piezas alrededor:

```
construir el lote  →  forward  →  loss  →  loss.backward()  →  optimizer.step()  →  zero_grad()
```

Ejecutamos 300 pasos. Si la teoría es correcta, la pérdida debe caer de forma claramente visible: al principio el modelo adivina casi al azar (pérdida ≈ ln(vocabulario)) y al final ha memorizado buena parte de la estructura repetitiva de los JSON.

In [5]:
# =====================================================================
# BUCLE DE ENTRENAMIENTO MANUAL — el esqueleto de todo entrenamiento
# =====================================================================
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)
STEPS = 300

for step in range(STEPS + 1):
    # 1. Construir el lote (datos + respuesta correcta)
    xb, yb = get_batch(batch_size=64)

    # 2. Forward: el modelo predice y calculamos la pérdida
    logits, loss = model(xb, yb)

    # 3. Backward: autograd rellena .grad de cada parámetro
    loss.backward()

    # 4. El optimizador mueve los parámetros en contra del gradiente
    optimizer.step()

    # 5. Limpiar los gradientes (PyTorch los acumula si no lo hacemos)
    optimizer.zero_grad()

    if step % 50 == 0:
        print(f"paso {step:4d}   loss {loss.item():.4f}")
# =====================================================================

paso    0   loss 5.6030


paso   50   loss 3.8679


paso  100   loss 2.8451


paso  150   loss 2.2553


paso  200   loss 1.6049


paso  250   loss 1.1348


paso  300   loss 1.0381


## 6. Lectura de resultados y sobreajuste

La pérdida cae de forma sostenida: el modelo pasó de adivinar a capturar la sintaxis repetitiva de los blueprints (llaves, comillas, claves como `"path"` o `"purpose"`, etc.).

**¿Dónde está el truco?** Entrenamos y medimos sobre *el mismo* texto de ~4.500 caracteres con un modelo de ~600 mil parámetros. Eso es memorización casi garantizada:

- **Sobreajuste (*overfitting*)** sería exactamente esto si lo dejáramos correr: la pérdida de *entrenamiento* seguiría bajando hasta casi 0 (el modelo recita el corpus), mientras que la pérdida sobre texto **nunca visto** dejaría de mejorar y empezaría a *subir*. El modelo aprende el examen, no la materia.
- Por eso el proyecto tiene splits congelados train/val/test (Fase 0): la única cifra honesta es la medida sobre datos que el modelo no entrenó.
- Aquí el sobreajuste no es un fallo, es la demostración: con tan poco texto, *ver* la pérdida caer prueba que el mecanismo (gradientes + optimizador) funciona. Que generalice ya es problema de datos y de escala, no del bucle.

Comprobación final: generamos texto carácter a carácter con el modelo entrenado, muestreando el siguiente carácter según las probabilidades del modelo (temperatura 0,8) en vez de quedarnos siempre con el más probable. No esperes blueprints válidos —espera fragmentos con forma de JSON mezclados con ruido, que es justo lo que un modelo que memorizó patrones locales debería producir.

In [6]:
model.eval()
torch.manual_seed(123)  # semilla solo para esta generación: salida reproducible
with torch.no_grad():
    # Semilla: los primeros BLOCK caracteres reales del corpus.
    generated = data[:BLOCK].unsqueeze(0).clone()
    for _ in range(268):
        logits, _ = model(generated[:, -BLOCK:])
        # logits tiene forma (batch, vocab_size): la fila 0 ES la distribución.
        probs = F.softmax(logits[0] / 0.8, dim=-1)  # temperatura 0,8
        next_id = torch.multinomial(probs, num_samples=1).unsqueeze(0)
        generated = torch.cat([generated, next_id], dim=1)

print("".join(itos[int(i)] for i in generated[0]))

{"blueprint_version": "1.0", "name": "llakfleoe cont"和t"\ Sudeto" q". "n neAe psatere"\ ",heW"s "oiAioites vel whive or 要oreseare fos atl.gidaede anemor w.r saeler atois ark要hedteriod ioy s) aor Soomeyse .Iise rch res an\hilinke wceta it afill pletboa"\ "jhcamenilskilyIp{uFlon"[ "\, poifirt-{emea.r 


## 7. Cierre: de este juguete a la Fase 3

- El bucle no cambia: QLoRA entrenará con **este mismo esqueleto**. Cambian la escala (7B parámetros en vez de ~0,6 M), los datos (582 registros filtrados en la Fase 0 en vez de 3 truncados) y *qué* parámetros reciben gradiente (solo los adaptadores LoRA del notebook 03).
- La diferencia entre "entrenar" y "hacer *fine-tuning*" no es el bucle: es **qué parámetros dejas libres** y con qué datos alimentas el `forward`.
- Si alguna vez una pérdida no baja, el diagnóstico empieza aquí: ¿llega gradiente a los parámetros (`.grad` no es `None` ni cero)? ¿El lr es razonable? ¿Estás limpiando los gradientes? Tres preguntas que este notebook te deja responder sin librerías de por medio.